# Implement with WGF

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import optuna
import jax
import jax.numpy as jnp
import optuna
import json
from functools import partial
from jax import jit, vmap, vmap
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import numpy as np

In [4]:
import sys
from pathlib import Path
# Add project root to path for imports
ROOT = Path.cwd().parent  
sys.path.append(str(ROOT))

from src.gradient_flows.optimize import objective, calculate_losses
from src.data_io.maps import load_maps_npz
from src.gradient_flows.utils import extract_quality_trajectories, parse_df_row, prepare_play_data
from src.gradient_flows.viz_tools import run_simulation, create_interactive_plot


ImportError: cannot import name 'calculate_losses' from 'src.gradient_flows.optimize' (/Users/cadenp/Documents/GitHub/DSC180B_FinalProject/src/gradient_flows/optimize.py)

In [5]:
%load_ext autoreload
%autoreload 2

## Optimize Variables

In [6]:
df = pd.read_parquet("../data/processed/def_features_test/defense_01.18.2016.GSW.at.CLE_21500622.parquet")
df = df.drop(index=[29, 84], errors='ignore')
df = df.reset_index(drop=True)

In [7]:
df.head()

,GAME_ID,SHOT_EVENT_ID,tracking_event_id,event_list_idx,PERIOD,game_clock,PLAYER_ID,TEAM_ID,flipped_coordinates,ball_x_traj,...,off5_q_traj,def5_pid,def5_x_traj,def5_y_traj,release_frame_global_idx,pbp_frame_global_idx,local_release_idx,local_pbp_idx,ball_z_traj,SHOT_MADE_FLAG
0,21500622,8,1,0,1,617,201567,1610612739,0,"[20.40327, 20.27989, 20.16353, 20.0552, 19.955...",...,"[0.8042342662811279, 0.8042342662811279, 0.804...",203084,"[-0.515509999999999, -0.7155699999999996, -0.9...","[3.01079, 2.89893, 2.7879500000000004, 2.68778...",50,149,50,149,"[3.54135, 3.6859, 3.80584, 3.89686, 3.95463, 3...",0
1,21500622,10,2,1,1,610,202691,1610612744,1,"[-12.015419999999999, -12.03564, -12.323990000...",...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",202389,"[-2.5867499999999986, -2.617609999999999, -2.6...","[21.301969999999997, 21.106830000000002, 20.90...",258,334,75,151,"[8.92556, 9.33024, 9.70121, 10.03012, 10.08909...",1
2,21500622,28,17,4,1,516,2544,1610612739,0,"[4.926300000000001, 5.964169999999999, 7.05731...",...,"[1.3832465410232544, 1.3832465410232544, 1.417...",203084,"[0.6417499999999983, 0.6631099999999996, 0.676...","[11.754650000000002, 11.429739999999999, 11.11...",271,359,75,163,"[7.00069, 6.94837, 6.91019, 6.84764, 6.66811, ...",0
3,21500622,30,20,7,1,503,203110,1610612744,1,"[-12.307899999999997, -12.209679999999999, -12...",...,"[1.1572421789169312, 1.1572421789169312, 1.157...",202389,"[1.4192800000000005, 1.4939299999999989, 1.577...","[7.067409999999995, 7.102639999999994, 7.14706...",241,309,75,143,"[4.38831, 4.63916, 4.61552, 4.80345, 4.82673, ...",0
4,21500622,34,21,8,1,486,101106,1610612744,1,"[-20.913269999999997, -20.941429999999997, -20...",...,"[1.140377402305603, 1.140377402305603, 1.14037...",202389,"[-7.172840000000001, -7.26829, -7.350349999999...","[14.531360000000006, 14.549440000000004, 14.57...",451,520,75,144,"[3.68183, 3.60123, 3.5027, 3.38264, 3.21821, 3...",0


In [6]:
STUDY_NAME = "nba-defensive-optimization-v6-final"
STORAGE_NAME = f"sqlite:///{STUDY_NAME}.db"

print("Starting optimization study...")

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_NAME,
    directions=["minimize", "minimize"],
    load_if_exists=True # Set to False for the first run!
)

# Run 100 trials (adjust this number based on how much time you have)
study.optimize(lambda trial: objective(trial, df), n_trials=20)

print("\nOptimization Finished!")
print(f"Total trials recorded: {len(study.trials)}")

# Print the best trade-offs (Pareto Front)
print("\nPareto Front (Best Trials):")
for trial in study.best_trials:
    print(f"  Trial {trial.number}:")
    print(f"  Losses (Pressure, Smoothness): {trial.values}")
    
# Try to visualize the Pareto front
try:
    import optuna.visualization as vis
    fig = vis.plot_pareto_front(study)
    fig.show()
    
except ImportError:
    print("Skipping visualization. (Install plotly/kaleido to view the Pareto front graph).")

[I 2026-02-28 06:00:43,051] Using an existing study with name 'nba-defensive-optimization-v6-final' instead of creating a new one.


Starting optimization study...


[I 2026-02-28 06:00:43,581] Trial 12 finished with values: [4.29416036605835, 0.04127800092101097] and parameters: {'max_acceleration': 72.36775126622419, 'velocity_cap': 15.8680696246732, 'learning_rate': 0.02025842843340893, 'jko_num_steps': 36, 'cushion_dist': 1.552377510901168, 'attraction_weight': 17.036813035910654, 'field_weight': -49.197699051929824, 'basket_gravity_weight': 9.148699539530913, 'global_ball_weight': 0.032547652405101504, 'sigma_long': 9.727795963626564, 'sigma_wide': 3.054752841177297, 'lane_blocking_weight': 32.43758980788306, 'occupancy_weight': 41.77696558953873, 'cohesion_weight': 1.2057906059199797, 'formation_radius': 15.87179653899587}.
[I 2026-02-28 06:00:44,086] Trial 13 finished with values: [4.147790908813477, 0.19622451066970825] and parameters: {'max_acceleration': 43.183105800761496, 'velocity_cap': 18.077292233749855, 'learning_rate': 0.03055814163369148, 'jko_num_steps': 28, 'cushion_dist': 1.6836643470113095, 'attraction_weight': 9.2973550511544


Optimization Finished!
Total trials recorded: 32

Pareto Front (Best Trials):
  Trial 1:
  Losses (Pressure, Smoothness): [4.230434417724609, 0.02236351929605007]
  Trial 3:
  Losses (Pressure, Smoothness): [4.096995830535889, 0.2040349841117859]
  Trial 5:
  Losses (Pressure, Smoothness): [2.5768930912017822, 0.34914711117744446]
  Trial 8:
  Losses (Pressure, Smoothness): [2.4695544242858887, 0.3535440266132355]
  Trial 9:
  Losses (Pressure, Smoothness): [2.428743600845337, 0.36457180976867676]
  Trial 10:
  Losses (Pressure, Smoothness): [2.26155424118042, 0.3787471354007721]
  Trial 13:
  Losses (Pressure, Smoothness): [4.147790908813477, 0.19622451066970825]
  Trial 15:
  Losses (Pressure, Smoothness): [2.988530158996582, 0.317974328994751]
  Trial 16:
  Losses (Pressure, Smoothness): [2.675170421600342, 0.34086912870407104]
  Trial 19:
  Losses (Pressure, Smoothness): [2.8217051029205322, 0.3324817419052124]
  Trial 21:
  Losses (Pressure, Smoothness): [3.555518627166748, 0.267